# Try ReFactX

## Setup
Create a `.env` file in the notebooks folder adding the following variable:
```
HTTP_BASE_URL="http://{user}:{password}@{host}:{port}/{dbname}"
```
Then append `/tablename` for using a specific db table.

(you can also set the HTTP_BASE_URL in the environment)

In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import os
from dotenv import load_dotenv

import torch
import time
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from transformers import AutoProcessor, AutoModelForImageTextToText

import refactx

In [5]:
load_dotenv()
HTTP_BASE_URL = os.environ.get('HTTP_BASE_URL')

In [14]:
tablename = 'qwen36'
HTTP_URL = f'{HTTP_BASE_URL}/{tablename}'

In [6]:
MODEL = 'Qwen/Qwen3.5-2B'
IS_VLM = True

In [7]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cpu'

In [11]:
if IS_VLM:
    processor = AutoProcessor.from_pretrained(MODEL)
    model = AutoModelForImageTextToText.from_pretrained(MODEL, device_map='auto')
    tokenizer = processor
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(MODEL, device_map='auto')

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
/opt/conda/envs/trl/lib/python3.14/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
Loading weights: 100%|██████████| 617/617 [00:00<00:00, 1309.87it/s]


In [15]:
index = refactx.load_index(
    HTTP_URL, 
    #tokenizer,
    #configkey=-200,
    #cache='simple'
)

Applying index config...


In [16]:
index.get_config()

Applying index config...


{'switch_parameter': 7, 'rootkey': -100, 'tokenizer_name': 'Qwen/Qwen3.6-27B'}

In [17]:
streamer = TextStreamer(tokenizer)

In [19]:
question = 'Is Johnny Depp older than Brad Pitt?'

prompted_texts = [refactx.apply_prompt_template(tokenizer, question=question)]

In [21]:
inputs = tokenizer.tokenizer(prompted_texts, return_tensors='pt', padding=True, padding_side='right')
inputs = inputs.to(model.device)
print(inputs['input_ids'].shape)

torch.Size([1, 780])


In [22]:
model.device

device(type='cpu')

In [23]:
# no need for num_beams=1
#refactx.patch_model(model)

In [24]:
num_beams = 1
num_batches = 1

auto_streamer = streamer if num_beams == 1 else None

In [25]:
constrained_processor = refactx.get_constrained_logits_processor(tokenizer, index, num_beams, num_batches)

In [26]:
logits_processor_list = constrained_processor

model.eval()
start = time.time()

with torch.no_grad():
    out = model.generate(
        **inputs,
        logits_processor=logits_processor_list,
        max_new_tokens=800,
        streamer = auto_streamer,
        do_sample = False,
        temperature = None,
        top_k=None,
        num_beams=num_beams,
        num_return_sequences=num_beams,
        use_cache=True,
        top_p=None,
        min_p=None,
    )

print('Elapsed', time.time() - start)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


<|im_start|>system
You are a helpful question-answering assistant that bases its answers on facts from a knowledge base and always respects the prompt.

The process to answer questions:

    You receive an input question.

    You determine the reasoning path needed to answer the question based on the information available.

    You determine the kind of answer you are asked. It can be a yes/no, a single entity, or a list of entities. Pay attention to the questions whose answer is a list of entities (e.g. Which countries share a border with Spain?): you need to find all the answer entities and include them all in the final answer.

    You get relevant facts with the "Fact:" command. You can rely on these facts and use them a proof for your answer.
    While getting facts you continue the reasoning explaining it step by step.

    Often description or short description may be useful for answering questions.

    You conclude with a concise answer that depending on the question can be a

### Visualize ReFactX output

In [27]:
_from = len(inputs.input_ids[0]) # 0
for i in range(out.shape[0]):
    print('-'*30, sum(out[i][_from:]), len(out[i][_from:]))
    print(tokenizer.decode(out[i][_from:]))

------------------------------ tensor(1299391) 192
Reasoning: To determine if Johnny Depp is older than Brad Pitt, I need to find their birth dates and compare them.
Let's start with Johnny Depp description.
Fact: <Johnny Depp> <date of birth> <1963-06-09T00:00:00Z> .
I found proof that Johnny Depp was born on June 9, 1963. Now I need the birth date of Brad Pitt.
Fact: <Brad Pitt> <date of birth> <1963-12-18T00:00:00Z> .
I found proof that Brad Pitt was born on December 18, 1963. Comparing the two dates, Johnny Depp was born in June and Brad Pitt in December, meaning Johnny Depp is older.

Answer: Yes<|im_end|>
<|endoftext|>


### Generated Facts

In [28]:
for i, triple in enumerate(refactx.get_constrained_states()[0][0].generated_triples):
    print(i, tokenizer.decode(triple), end='\n')

0  <Johnny Depp> <date of birth> <1963-06-09T00:00:00Z> .
1  <Brad Pitt> <date of birth> <1963-12-18T00:00:00Z> .
